# Aerosol encoder long SGP training

This notebook trains the no-VAE 64-D aerosol encoder on Colab CUDA. It expects the project folder at `MyDrive/encode_aerosol` and the feature store under `artifacts/temporal_gru_30min_20161129_20230421/features/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

DRIVE_PROJECT = Path('/content/drive/MyDrive/encode_aerosol')
LOCAL_PROJECT = Path('/content/encode_aerosol')
LOCAL_FEATURE_DIR = Path('/content/aerosol_features/features')
DRIVE_FEATURE_DIR = DRIVE_PROJECT / 'artifacts/temporal_gru_30min_20161129_20230421/features'

assert DRIVE_PROJECT.exists(), f'Missing Drive project folder: {DRIVE_PROJECT}'
os.chdir('/content')

shutil.rmtree(LOCAL_PROJECT, ignore_errors=True)
shutil.rmtree(LOCAL_FEATURE_DIR.parent, ignore_errors=True)
subprocess.run(['rsync', '-a', '--exclude', 'artifacts', f'{DRIVE_PROJECT}/', f'{LOCAL_PROJECT}/'], check=True)
LOCAL_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
features_npz = LOCAL_FEATURE_DIR / 'features.npz'
shutil.copy2(DRIVE_FEATURE_DIR / 'features.npz', features_npz)
shutil.copy2(DRIVE_FEATURE_DIR / 'metadata.json', LOCAL_FEATURE_DIR / 'metadata.json')
shutil.copy2(DRIVE_FEATURE_DIR / 'feature_coverage.csv', LOCAL_FEATURE_DIR / 'feature_coverage.csv')
with zipfile.ZipFile(features_npz) as archive:
    archive.extract('X.npy', LOCAL_FEATURE_DIR)
    archive.extract('times.npy', LOCAL_FEATURE_DIR)
print('ready:', LOCAL_PROJECT, LOCAL_FEATURE_DIR)

In [ ]:
%cd /content/encode_aerosol
import subprocess
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'pyyaml', 'pandas', 'numpy', 'matplotlib', 'scikit-learn'], check=True)

In [ ]:
import torch

print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Change Colab runtime to GPU before training.'

In [ ]:
FEATURES = '/content/aerosol_features/features/features.npz'
CONFIG = 'configs/sgp_e13_no_htdma_30min_temporal_gru_64_bottleneck_no_vae_long_e13_70_15_15.yaml'
DRIVE_OUTPUT = '/content/drive/MyDrive/encode_aerosol/artifacts/temporal_gru_30min_20161129_20230421/run_det64_no_vae_70_15_15_colab_cuda'

subprocess.run([
    'python', '-m', 'aerosol_encoding.train',
    '--config', CONFIG,
    '--features', FEATURES,
    '--output', DRIVE_OUTPUT,
    '--device', 'cuda',
], check=True)

In [ ]:
from pathlib import Path
history = Path(DRIVE_OUTPUT) / 'history.csv'
assert history.exists(), f'Training did not create {history}; inspect the training cell above.'
subprocess.run([
    'python', '-m', 'aerosol_encoding.plot_training',
    '--history', str(history),
    '--output', f'{DRIVE_OUTPUT}/training_curve.png',
    '--title', '64-D no-VAE long SGP aerosol encoder, Colab CUDA',
], check=True)

In [ ]:
checkpoint = Path(DRIVE_OUTPUT) / 'checkpoint.pt'
assert checkpoint.exists(), f'Training did not create {checkpoint}; inspect the training cell above.'
subprocess.run([
    'python', '-m', 'aerosol_encoding.evaluate_cross_prediction',
    '--features', FEATURES,
    '--checkpoint', str(checkpoint),
    '--output', f'{DRIVE_OUTPUT}/cross_prediction_test',
    '--split', 'test',
    '--batch-size', '512',
    '--device', 'cuda',
], check=True)